# QRadiomics — LUNG1 (NSCLC-Radiomics) survival reproduction

End-to-end reproduction of the Aerts *et al.* (Nature Communications 2014) radiomics
survival workflow on the TCIA **NSCLC-Radiomics (LUNG1)** cohort, driven entirely by
the `qr` CLI:

```
qr tcia download  ->  qr convert dicom-series / rtstruct (--roi GTV-1)
  ->  qr convert manifest-from-dir  ->  qr extract -p nsclc-survival
  ->  qr results merge  ->  qr analyze survival
```

Runs on a configurable patient subset so it finishes inside a Colab session.
Every comment in this notebook is in English.

## 0. Install qradiomics

In [ ]:
# Install qradiomics into the Colab runtime using the official kickoff script.
# QR_VENV=- installs into the current environment (no venv) so `qr` is on PATH.
!curl -sSL https://raw.githubusercontent.com/choilab-jefferson/qradiomics/main/scripts/kickoff.sh | QR_VENV=- bash

# rt-utils is required by `qr convert rtstruct` and is an optional extra.
!pip install -q rt-utils

# Verify the CLI is available.
!qr info

## 1. Download a paired subset from TCIA

We keep only patients that have **both** a CT series and an RTSTRUCT, then take a
subset. Bump `N_PATIENTS` up for a stronger survival signal (the full cohort is 422).

In [ ]:
import os
import pandas as pd

N_PATIENTS = 60  # subset size; increase for more statistical power

# 1a. List every series in the collection.
!qr tcia series --collection NSCLC-Radiomics -o /content/series_list.csv
df_series = pd.read_csv('/content/series_list.csv')

# 1b. Keep patients that have both a CT and an RTSTRUCT series.
pats_with_ct = set(df_series[df_series['Modality'] == 'CT']['PatientID'])
pats_with_rt = set(df_series[df_series['Modality'] == 'RTSTRUCT']['PatientID'])
target_pats = sorted(pats_with_ct & pats_with_rt)[:N_PATIENTS]
print(f'Selected {len(target_pats)} paired patients.')

# 1c. Download only the CT + RTSTRUCT series for the selected patients.
df_target = df_series[df_series['PatientID'].isin(target_pats) &
                      df_series['Modality'].isin(['CT', 'RTSTRUCT'])]
df_target.to_csv('/content/target_manifest.csv', index=False)
!qr tcia download --manifest /content/target_manifest.csv -o /content/Lung1 -j 8
print('Download complete.')

## 2. Convert DICOM to NRRD

Each CT series becomes `{pid}_image.nrrd`; each RTSTRUCT becomes `{pid}_mask.nrrd`.

**Important:** we pass `--roi GTV-1` explicitly. Without it, `qr convert rtstruct`
falls back to the *first* ROI in the structure set, which for LUNG1 is often a lung
or cord contour rather than the tumour — silently producing the wrong mask.

In [ ]:
import os
import pandas as pd

os.makedirs('/content/Lung1-out', exist_ok=True)
df_target = pd.read_csv('/content/target_manifest.csv')
df_ct = df_target[df_target['Modality'] == 'CT']
df_rt = df_target[df_target['Modality'] == 'RTSTRUCT']
print(f'{len(df_ct)} CT series, {len(df_rt)} RTSTRUCT series. Converting...')

# 2a. CT series -> {pid}_image.nrrd
for _, row in df_ct.iterrows():
    pid = row['PatientID']
    d_path = f"/content/Lung1/{pid}/{row['StudyInstanceUID']}/{row['SeriesInstanceUID']}"
    out_path = f'/content/Lung1-out/{pid}_image.nrrd'
    if not os.path.exists(out_path) and os.path.exists(d_path):
        !qr convert dicom-series -i {d_path} -o {out_path}

# 2b. RTSTRUCT -> {pid}_mask.nrrd, always extracting the GTV-1 tumour ROI.
#     Only attempt this when the CT step actually produced {pid}_image.nrrd.
#     A few LUNG1 CT series are not GDCM-readable (e.g. LUNG1-035); skipping
#     them here avoids a failed rtstruct conversion on a missing reference CT.
for _, row in df_rt.iterrows():
    pid = row['PatientID']
    ct_image = f'/content/Lung1-out/{pid}_image.nrrd'
    if not os.path.exists(ct_image):
        print(f'Skipping {pid}: CT was not converted (no {pid}_image.nrrd).')
        continue
    rt_path = f"/content/Lung1/{pid}/{row['StudyInstanceUID']}/{row['SeriesInstanceUID']}"
    out_path = f'/content/Lung1-out/{pid}_mask.nrrd'
    ct_row = df_ct[df_ct['PatientID'] == pid]
    if ct_row.empty:
        continue
    ct_dir = f"/content/Lung1/{pid}/{ct_row.iloc[0]['StudyInstanceUID']}/{ct_row.iloc[0]['SeriesInstanceUID']}"
    if not os.path.exists(out_path) and os.path.exists(rt_path) and os.path.exists(ct_dir):
        print(f'Converting GTV-1 for {pid}...')
        !qr convert rtstruct -d {ct_dir} -r {rt_path} --roi GTV-1 -o {out_path}

## 3. Build the manifest

Pair each `{pid}_image.nrrd` with its `{pid}_mask.nrrd` into a manifest CSV. We build
it in Python here so the step is independent of the installed qradiomics version.

> `qr convert manifest-from-dir -d /content/Lung1-out --image-glob '*_image.nrrd'
> --mask-glob '*_mask.nrrd' -o /content/Lung1-out/manifest.csv` does the same on
> recent qradiomics. If you prefer the CLI and hit *"No image/mask pairs found"*,
> your Colab is reusing an older clone — force a refresh with
> `!rm -rf /content/qradiomics` and re-run the install cell.

In [ ]:
import os
import glob
import pandas as pd

rows = []
for img in sorted(glob.glob('/content/Lung1-out/*_image.nrrd')):
    pid = os.path.basename(img)[:-len('_image.nrrd')]
    mask = f'/content/Lung1-out/{pid}_mask.nrrd'
    if os.path.exists(mask):
        rows.append({'patient_id': pid, 'modality': 'CT',
                     'image_path': img, 'mask_path': mask})

manifest = pd.DataFrame(rows)
manifest.to_csv('/content/Lung1-out/manifest.csv', index=False)
print(f'{len(manifest)} image/mask pairs.')
manifest.head()

## 4. Extract radiomics features

The bundled `nsclc-survival` pattern reproduces the Aerts feature space
(Original + LoG + Wavelet + Square + SquareRoot + Logarithm over 7 feature classes,
~1130 features per patient).

In [ ]:
# -j sets parallel workers. Large CT volumes (e.g. 500x500x300) are memory
# -heavy under the wavelet/LoG filters; on a low-RAM runtime a worker can be
# OOM-killed. qr extract keeps the patients that finished and reports the
# rest -- if you see 'worker process died (out of memory)', re-run with -j 1
# (slower, lower peak memory) to pick up the remainder.
!qr extract \
    -m /content/Lung1-out/manifest.csv \
    -p nsclc-survival \
    -o /content/features.csv \
    -j 2

import pandas as pd
features_df = pd.read_csv('/content/features.csv')
print(f'{features_df.shape[0]} patients x {features_df.shape[1] - 1} features')
features_df.head()

## 5. Merge with clinical survival data

Download the published LUNG1 clinical table and join it on `PatientID`. In survival
mode `qr results merge` auto-converts the survival time to months and emits two
canonical columns: **`OS_months`** and **`OS_event`**.

In [ ]:
!curl -o /content/clinical_data.csv -sSL \
    https://www.cancerimagingarchive.net/wp-content/uploads/NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv

!qr results merge \
    -f /content/features.csv \
    -c /content/clinical_data.csv \
    --clinical-id-col PatientID \
    --time-col Survival.time \
    --event-col deadstatus.event \
    -o /content/analysis_ready.csv

import pandas as pd
final_df = pd.read_csv('/content/analysis_ready.csv')
print(f'Merged {len(final_df)} patients.')
final_df[['patient_id', 'OS_months', 'OS_event']].head()

## 6. Cox proportional-hazards survival analysis

Univariate Cox PH is run on every radiomic feature against overall survival. The merge
step standardised the outcome columns, so we point `qr analyze survival` straight at
`OS_months` / `OS_event` (no column guessing needed).

In [ ]:
!qr analyze survival \
    -i /content/analysis_ready.csv \
    --outcome OS_months --event OS_event \
    -o /content/cox_results.csv \
    --top-n 20

import pandas as pd
cox_df = pd.read_csv('/content/cox_results.csv')
print(f'{len(cox_df)} features fitted; {(cox_df.p < 0.05).sum()} significant at p<0.05.')
cox_df.head(15)

## 7. Notes

- On the full 422-patient cohort, prognostic NGTDM / first-order texture features rank
  at the top, replicating the headline Aerts 2014 result. On a small subset the exact
  ranking will vary — raise `N_PATIENTS` for a more faithful reproduction.
- `qr extract` without `-p` enables *all* image types and feature classes (~1400
  features, e.g. `wavelet-*`, `log-sigma-*`), not just `original_*`. Use a pattern
  (here `nsclc-survival`) for the curated, paper-aligned feature set.
- Citations: Aerts HJWL *et al.*, Nature Communications 2014; 5:4006. PyRadiomics:
  van Griethuysen JJM *et al.*, Cancer Research 2017; 77(21):e104-e107.